# 扫描器参数实验
本 Notebook 的实验计算只读取本地行情。默认建立独立合成示例库；真实模式请先运行 Notebook 01 的数据更新。

In [ ]:
from selectstock.runtime import check_runtime
check_runtime()
from selectstock.progress import DownloadProgress
from pathlib import Path
from copy import deepcopy
import pandas as pd
from IPython.display import display
from selectstock.config import load_config, save_experiment
from selectstock.demo import create_demo_config
from selectstock.runner import update_data, run_scans, load_bars, export_watchlists, check_data
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs" / "scanners.yaml").exists())

In [ ]:
DEMO = True
base = create_demo_config(ROOT) if DEMO else load_config(ROOT)
base["data"]["offline"] = True
experiment = deepcopy(base)
# 修改单个阈值便于观察因果；所有比较符和指标窗口见 scanners.yaml。
experiment["scanners"]["leader"]["min_week_turnover"] = 1_000_000_000
experiment["scanners"]["potent"]["min_volatility"] = 7.0
experiment["scanners"]["potent"]["excluded_industries"] = []
experiment["scanners"]["leader"]["enabled"] = True
experiment["scanners"]["potent"]["enabled"] = True

In [ ]:
baseline = run_scans(base, persist=False)
changed = run_scans(experiment, persist=False)
for name in baseline.frames:
    before = set(baseline.selected(name)["symbol"])
    after = set(changed.selected(name)["symbol"])
    print(name, "基准:", len(before), "实验:", len(after), "新增:", sorted(after-before), "移除:", sorted(before-after))
    display(changed.frames[name])

In [ ]:
# 每项条件独立淘汰数量；同一股票可能同时不满足多个条件。
for name, frame in changed.frames.items():
    checks = [c for c in frame if c.startswith("check_")]
    summary = pd.DataFrame([{"condition": c.removeprefix("check_"),
                             "passed": int(frame[c].eq(True).sum()),
                             "failed": int(frame[c].eq(False).sum()),
                             "unknown": int(frame[c].isna().sum())} for c in checks])
    print(name)
    display(summary)

In [ ]:
# 检查单股所有指标及未入选原因。
INSPECT_SYMBOL = base["data"]["symbols"][0] if base["data"]["symbols"] else "AAPL"
for name, frame in changed.frames.items():
    print(name, INSPECT_SYMBOL)
    display(frame.loc[frame["symbol"].eq(INSPECT_SYMBOL)].T)

In [ ]:
# 保存命名实验配置，后续通过 CLI --config 或 load_experiment 重现。
experiment_path = ROOT / "runs" / "experiments" / "liquidity_volatility.yaml"
save_experiment(experiment, experiment_path)
print(experiment_path)

历史日期扫描只允许使用当时已保存的证券资料快照；无法取得历史市值时显示资料不足。行业默认不排除，因为原始截图未公开完整行业清单。参数实验不自动更新行情，也不计算买卖收益。